# Solution E1 · The income in euros that wipes out the amount/income ratio

**Type**: Find the bug

## Objective

Understand why a regularized logistic changes its predictions if you change the **unit of measure** of a column (euros versus thousands of euros), find out which coefficient gets sacrificed and fix it with
a scaler inside the pipeline. Verify that after the fix the model is **invariant** to the units.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.3, stratify=df["defaulted"], random_state=0)


def evaluate(model, k=1.0):
    """Test AUC of a copy of the model, with `income` divided by k (k = 1000: thousands of euros)."""
    a, b = X_tr.copy(), X_te.copy()
    a["income"] /= k
    b["income"] /= k
    return roc_auc_score(y_te, clone(model).fit(a, y_tr).predict_proba(b)[:, 1])

## Commented solution

In [ ]:
bug_model = LogisticRegression(C=0.1, max_iter=1000)
auc_bug = evaluate(bug_model)                 # income in euros
auc_bug_k_euro = evaluate(bug_model, 1000)    # same data, income in thousands of euros
print(f"test AUC: {auc_bug:.3f} with the income in euros | {auc_bug_k_euro:.3f} in thousands of euros")

**Where the bug is.** The L2 penalty sums the squares of the coefficients. With the income in euros (scale 30,000) its coefficient can be tiny and still weigh what it needs to, so it is barely
penalized; the ratio (scale 0.5) needs a large coefficient to count and the penalty **squashes** it (`C = 0.1` is very strong). The model relies almost only on the income. With the income in
thousands of euros the balance changes: **same data, different model**.

In [ ]:
unit_difference = auc_bug_k_euro - auc_bug
bug_ratio_coef = clone(bug_model).fit(X_tr, y_tr).coef_[0][0]
print(f"difference between the two units: {unit_difference:.3f} | coefficient of the ratio in the colleague's model: {bug_ratio_coef:.4f}")

In [ ]:
ok_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000))
auc_ok = evaluate(ok_model)
auc_ok_k_euro = evaluate(ok_model, 1000)
invariance = abs(auc_ok - auc_ok_k_euro)
ok_model.fit(X_tr, y_tr)
ok_ratio_coef = ok_model[-1].coef_[0][0] / ok_model[0].scale_[0]
print(f"AUC with the scaler: {auc_ok:.4f} (euros) | {auc_ok_k_euro:.4f} (thousands of euros) | gap {invariance:.1e}")
print(f"coefficient of the ratio in original units: {ok_ratio_coef:.2f} (against {bug_ratio_coef:.4f} for the colleague)")

## Check

In [ ]:
with verify():
    check(auc_bug < 0.75, "the colleague's model with the income in euros is poor (AUC below 0.75)", "don't modify the colleague's code")
    check(close(unit_difference, auc_bug_k_euro - auc_bug) and unit_difference > 0.04, "changing only the unit of measure the AUC changes by over 0.04", "auc_bug_k_euro - auc_bug")
    check(bug_ratio_coef is not None and abs(bug_ratio_coef) < 0.02, "in the colleague's model the coefficient of the ratio has been squashed (below 0.02)", "coef_[0][0] of the model trained on the training set in euros")
    check(invariance is not None and invariance < 1e-3 and auc_ok > 0.77, "with the scaler the AUC does not depend on the unit of measure (gap below 0.001) and is 0.77 or more", "StandardScaler before LogisticRegression")
    check(ok_ratio_coef > 50 * abs(bug_ratio_coef), "in original units the right coefficient of the ratio is hundreds of times larger than the colleague's", "coef of the model / scaler.scale_[0]")
done()

## What you learned

- **A regularized logistic is not invariant to the scale of the features.** The penalty treats the coefficients as if they were comparable: they are so only if the features have the same scale. Here the coefficient of the ratio
  in the colleague's model is 0.008; in original units, with the scaler, it is 1.96: the penalty had reduced it by over 200 times.
- The bug raises no errors: the colleague's AUC (0.716) is "plausible" and sits below the correct one (0.778) by 0.06. You find it only with an **invariance** test: change the unit of a column and check that the predictions
  do not change. With the scaler the invariance is exact (gap below 1e-3), because standardization erases the unit.
- The scaler goes in the `Pipeline`: that way it is learned on the training part of every fold alone (M3) and you apply it identically to the test set. Scaling is not only for solvers: it serves to make the penalty meaningful.
- To interpret the coefficients in original units, divide them by the standard deviation of the scaler (`coef / scaler.scale_`), as you did here.

## Common pitfall

Regularizing (`Ridge`, `Lasso`, `LogisticRegression`, `SVC`, ...) without standardizing, "because the features are all positive anyway" or "because the model converges all the same". Convergence has nothing to do with it: it is the penalty
that depends on the scale.

## Interview question

<details><summary>Which models require feature scaling and which don't?</summary>

Those that use distances (KNN, kernel SVM, k-means), PCA and the **regularized** linear ones or those trained by gradient descent (logistic, Ridge, Lasso, networks). Trees and their ensembles (random forest, boosting)
use thresholds and are invariant to monotonic transformations of the individual features, so they don't need it. When in doubt I run an invariance test: I change the unit of a column and see whether the predictions change.

</details>